# AI Trading Model Training

This notebook provides GPU access for training the variable-length LSTM trading model.

**IMPORTANT**: This notebook contains MINIMAL code. All model logic is in `../src/` modules.

In [ ]:
import sys
sys.path.append('../src')

import torch
from torch.utils.data import DataLoader

from config import get_config
from utils import set_seed, get_device, count_parameters
from data import load_preprocessed_datasets, collate_variable_length
from models import VariableLengthLSTMTrader
from training import train_model
from evaluation import evaluate_model, comprehensive_backtest

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()
print(f"Using device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 1. Load and Prepare Data

In [ ]:
# Load configuration
config = get_config()

# Load preprocessed datasets (much faster than processing from raw CSV)
print("Loading preprocessed datasets...")
train_dataset, val_dataset, test_dataset, metadata = load_preprocessed_datasets(
    processed_dir='../data/processed'
)

print(f"\nDataset loaded successfully!")
print(f"Train samples: {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"\nTokens: {metadata['train_tokens']} train, {metadata['val_tokens']} val, {metadata['test_tokens']} test")

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    collate_fn=collate_variable_length,
    num_workers=config['dataloader']['num_workers'],
    pin_memory=config['dataloader']['pin_memory'],
    persistent_workers=config['dataloader']['persistent_workers'],
    prefetch_factor=config['dataloader']['prefetch_factor'],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    collate_fn=collate_variable_length,
    num_workers=config['dataloader']['num_workers'],
    pin_memory=config['dataloader']['pin_memory'],
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    collate_fn=collate_variable_length,
)

print("Data loaders created successfully")

## 2. Initialize Model

In [ ]:
# Initialize model
model = VariableLengthLSTMTrader(
    input_size=config['model']['input_size'],
    hidden_size=config['model']['hidden_size'],
    num_layers=config['model']['num_layers'],
    num_classes=config['model']['num_classes'],
    dropout=config['model']['dropout']
)

# Move to GPU
model = model.to(device)

# Print model info
print(f"Model parameters: {count_parameters(model):,}")
print(f"\nModel architecture:")
print(model)

## 3. Train Model

In [ ]:
# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    checkpoint_dir='../models/checkpoints'
)

print("\nTraining completed!")
print(f"Best validation loss: {history['best_val_loss']:.4f}")
print(f"Best epoch: {history['best_epoch'] + 1}")

## 4. Evaluate Model

In [ ]:
# Load best model
from utils import load_checkpoint

checkpoint = load_checkpoint(
    '../models/best_model.pth',
    model,
    device=device
)

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

In [ ]:
# Evaluate on test set
metrics = evaluate_model(
    model=model,
    test_loader=test_loader,
    device=device,
    confidence_threshold=config['trading']['confidence_threshold']
)

print("\nTest Set Metrics:")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Average Confidence: {metrics['avg_confidence']:.4f}")
print(f"\nPer-Class Metrics:")
print(f"Precision: {metrics['precision']}")
print(f"Recall: {metrics['recall']}")
print(f"F1 Score: {metrics['f1_score']}")
print(f"\nConfusion Matrix:")
print(metrics['confusion_matrix'])

## 5. Comprehensive Backtest

In [ ]:
# Run comprehensive backtest
backtest_results = comprehensive_backtest(
    model=model,
    test_dataset=test_dataset,
    device=device,
    confidence_threshold=config['trading']['confidence_threshold']
)

print("\nBacktest Results:")
print(f"Total PnL: {backtest_results['total_pnl']:.4f} SOL")
print(f"Number of Tokens: {backtest_results['num_tokens']}")
print(f"Total Trades: {backtest_results['total_trades']}")
print(f"Overall Win Rate: {backtest_results['overall_win_rate']:.2%}")
print(f"Sharpe Ratio: {backtest_results['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {backtest_results['max_drawdown']:.2%}")
print(f"Avg Trades per Token: {backtest_results['avg_trades_per_token']:.2f}")
print(f"Avg PnL per Token: {backtest_results['avg_pnl_per_token']:.4f} SOL")

## 6. Save Final Results

In [ ]:
import json
from datetime import datetime

# Save results
results_summary = {
    'timestamp': datetime.now().isoformat(),
    'config': config,
    'training_history': {
        'best_epoch': int(history['best_epoch']),
        'best_val_loss': float(history['best_val_loss']),
    },
    'test_metrics': {
        'accuracy': float(metrics['accuracy']),
        'avg_confidence': float(metrics['avg_confidence']),
    },
    'backtest_results': {
        'total_pnl': float(backtest_results['total_pnl']),
        'num_tokens': int(backtest_results['num_tokens']),
        'total_trades': int(backtest_results['total_trades']),
        'win_rate': float(backtest_results['overall_win_rate']),
        'sharpe_ratio': float(backtest_results['sharpe_ratio']),
        'max_drawdown': float(backtest_results['max_drawdown']),
    }
}

with open('../models/results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results saved to ../models/results_summary.json")